# Topic 10 — K-Nearest Neighbors (KNN)
### Theory → tiny example → from-scratch implementation → sklearn → effect of k.

KNN is the simplest possible classifier: to predict a new point's class, look at its `k` closest
points in the training data and take a **majority vote**. No real "training" happens — it just
memorizes the data and does all the work at prediction time (this is called a "lazy learner").

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import make_classification
from collections import Counter

rng = np.random.default_rng(0)

## 1. Distance — Euclidean distance

The most common distance metric between two points:

```text
distance(a, b) = sqrt( (a1-b1)^2 + (a2-b2)^2 + ... )
```

This is literally the Pythagorean theorem, generalized to any number of dimensions.

In [ ]:
def euclidean_distance(a, b):
    return np.sqrt(np.sum((a - b) ** 2))

p1 = np.array([0, 0])
p2 = np.array([3, 4])
print("distance:", euclidean_distance(p1, p2))   # classic 3-4-5 triangle -> 5.0

plt.figure(figsize=(4, 4))
plt.plot([p1[0], p2[0]], [p1[1], p2[1]], marker="o")
plt.annotate("p1", p1, textcoords="offset points", xytext=(5, 5))
plt.annotate("p2", p2, textcoords="offset points", xytext=(5, 5))
plt.title(f"Euclidean distance = {euclidean_distance(p1, p2):.1f}")
plt.grid(True)
plt.show()

## 2. From-scratch KNN

Find the `k` nearest training points to a new point, then take a majority vote of their labels.

In [ ]:
def knn_predict(X_train, y_train, x_new, k=3):
    distances = [euclidean_distance(x_new, x) for x in X_train]
    nearest_idx = np.argsort(distances)[:k]       # indices of the k closest points
    nearest_labels = y_train[nearest_idx]
    vote = Counter(nearest_labels).most_common(1)[0][0]   # majority vote
    return vote, nearest_idx

X_toy = np.array([
    [1, 1], [1, 2], [2, 1],      # cluster A (label 0)
    [8, 8], [8, 9], [9, 8],      # cluster B (label 1)
])
y_toy = np.array([0, 0, 0, 1, 1, 1])

new_point = np.array([2, 2])
prediction, neighbor_idx = knn_predict(X_toy, y_toy, new_point, k=3)
print("predicted class:", prediction)
print("nearest neighbor indices used:", neighbor_idx)

plt.figure(figsize=(5, 5))
plt.scatter(X_toy[:, 0], X_toy[:, 1], c=y_toy, cmap="bwr", s=80, label="train")
plt.scatter(*new_point, c="green", marker="*", s=250, label="new point")
for idx in neighbor_idx:
    plt.plot([new_point[0], X_toy[idx, 0]], [new_point[1], X_toy[idx, 1]], "k--", lw=0.8)
plt.legend()
plt.title(f"KNN (k=3) predicts class {prediction}")
plt.show()

## 3. sklearn implementation

In [ ]:
X, y = make_classification(
    n_samples=200, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, class_sep=1.2, random_state=42
)

clf = KNeighborsClassifier(n_neighbors=5)
clf.fit(X, y)

sample = X[:5]
print("predictions:", clf.predict(sample))
print("true labels:", y[:5])
print("accuracy on training data:", clf.score(X, y))

## 4. Effect of `k` on the decision boundary

Small `k` (e.g. k=1): very flexible, sensitive to individual points and noise (can overfit).
Large `k`: smoother boundary, more stable, but can underfit and blur the boundary between classes.

In [ ]:
def plot_knn_boundary(ax, X, y, k):
    clf = KNeighborsClassifier(n_neighbors=k).fit(X, y)
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap="bwr")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="bwr", edgecolor="k")
    ax.set_title(f"k={k}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, k in zip(axes, [1, 5, 50]):
    plot_knn_boundary(ax, X, y, k)
plt.tight_layout()
plt.show()
# k=1  -> jagged, overfit boundary that chases individual noisy points
# k=5  -> reasonably smooth, balanced
# k=50 -> very smooth, may underfit and ignore local structure

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Implement knn_predict for k=1 and k=5 on the new_point [2,2] example above -- does the prediction change?
# 2. Use sklearn's KNeighborsClassifier with weights="distance" (closer neighbors count more) instead
#    of the default uniform voting, and compare its decision boundary to k=5 uniform above.
# 3. KNN uses Euclidean distance by default -- look up why this can behave badly on unscaled features
#    (hint: reread Topic 15, Feature Scaling, once you get there) and write one sentence about it.
# 4. Why is KNN usually a poor fit for high-dimensional sparse text features like TF-IDF vectors
#    (Topic 24)? (This is a "think about it now, understand it fully later" question.)

---
### Next up: **Topic 11 — Naive Bayes** (very important for NLP/text classification).

Say "next" when you're ready.